# vPCF Model Training on Google Colab

This notebook trains DEC/IDEC clustering models on vPCF data from HDF5 and DM3 files.

## Step 1: Setup Environment

In [1]:
# Check if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

import sys
import os

Running on Google Colab


In [2]:
# Install required packages
!pip install -q h5py hyperspy tqdm scikit-learn pandas tensorflow

In [ ]:
# Detect environment and setup paths
if IN_COLAB:
    # Check if we're in web Colab or VS Code Colab
    try:
        import google.colab.notebook
        IS_WEB_COLAB = True
        print("Running on Web Colab")
    except:
        IS_WEB_COLAB = False
        print("Running on VS Code Colab Extension")
    
    if IS_WEB_COLAB:
        # Web Colab: Can mount Google Drive
        from google.colab import drive
        drive.mount('/content/drive')
        project_dir = '/content/drive/My Drive/CMU vPCF Project'
        os.chdir(project_dir)
        print(f"Changed to: {os.getcwd()}")
        
        h5_file = '/content/drive/My Drive/CMU vPCF Project/Experimentally-obtained vPCF Testing/data/vPCF_test_2.h5'
        dm3_file = '/content/drive/My Drive/CMU vPCF Project/Experimentally-obtained vPCF Testing/data/vPCF_test_2.dm3'
    else:
        # VS Code Colab: Use upload directory or current working directory
        print("Note: Upload your data files to Colab using the file manager on the left")
        h5_file = './data/vPCF_test_2.h5'
        dm3_file = './data/vPCF_test_2.dm3'
        project_dir = os.getcwd()
        print(f"Current directory: {project_dir}")
else:
    # Local execution: Go to project root
    project_dir = r'C:\Users\alexg\Downloads\CMU vPCF Project'
    os.chdir(project_dir)
    print(f"Changed to: {os.getcwd()}")
    
    h5_file = r'Experimentally-obtained vPCF Testing\data\vPCF_test_2.h5'
    dm3_file = r'Experimentally-obtained vPCF Testing\data\vPCF_test_2.dm3'

Running on VS Code Colab Extension
Note: Upload your data files to Colab using the file manager on the left
Current directory: /content


In [4]:
# Add paths and verify imports
# For local execution, we need to add src/ from the project root
# The notebook is in Experimentally-obtained vPCF Testing/, so we go up one level
if not IN_COLAB:
    src_path = os.path.join(os.getcwd(), 'src')
    if os.path.exists(src_path):
        sys.path.insert(0, src_path)
        print(f"Added to sys.path: {src_path}")
    else:
        print(f"Warning: src path not found at {src_path}")
else:
    # For Colab, src/ should be in the current working directory
    sys.path.insert(0, './src')

# Verify imports
try:
    from vpcf_data_loader import load_vpcf_file, check_dependencies
    print("✓ Successfully imported vpcf_data_loader")
    
    # Check dependencies
    print("\nDependencies:")
    for name, available in check_dependencies().items():
        status = "✓ YES" if available else "✗ NO"
        print(f"  {name}: {status}")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print(f"\nCurrent working directory: {os.getcwd()}")
    print(f"sys.path entries:")
    for path in sys.path[:5]:
        print(f"  {path}")
    print("\nNote: Make sure you're in the correct directory or have uploaded the necessary files")

✗ Import error: No module named 'vpcf_data_loader'

Current working directory: /content
sys.path entries:
  ./src
  /content
  /env/python
  /usr/lib/python312.zip
  /usr/lib/python3.12

Note: Make sure you're in the correct directory or have uploaded the necessary files


## Step 2: Load and Preprocess Data

In [5]:
import numpy as np
from vpcf_data_loader import load_vpcf_file, combine_datasets

# File paths are already set in the setup cell
print(f"H5 file exists: {os.path.exists(h5_file)}")
print(f"DM3 file exists: {os.path.exists(dm3_file)}")

ModuleNotFoundError: No module named 'vpcf_data_loader'

In [ ]:
# Load H5 data with histogram features
# Use a smaller subset first for testing
print("Loading H5 data...")
h5_dataset = load_vpcf_file(
    h5_file,
    feature_method="histogram",  # "flatten", "histogram", "statistical", "combined"
    normalize="minmax",
    max_frames=500,  # Start with 500 frames for testing; remove for full dataset
    verbose=True
)

print(f"Loaded: {h5_dataset}")
print(f"Feature matrix shape: {h5_dataset.features.shape}")

In [ ]:
# Optionally load DM3 data
print("Loading DM3 data...")
dm3_dataset = load_vpcf_file(
    dm3_file,
    feature_method="histogram",
    normalize="minmax",
    verbose=True
)

print(f"Loaded: {dm3_dataset}")

In [ ]:
# Use H5 data (has more samples)
x = h5_dataset.features

print(f"Feature matrix shape: {x.shape}")
print(f"Feature statistics:")
print(f"  Min: {x.min():.4f}")
print(f"  Max: {x.max():.4f}")
print(f"  Mean: {x.mean():.4f}")

## Step 3: Train DEC Model

In [ ]:
from src.DEC import DEC

# Configuration
n_clusters = 10
hidden_dims = [500, 500, 2000]
dims = [x.shape[1]] + hidden_dims + [n_clusters]

print(f"Network architecture: {dims}")
print(f"Number of clusters: {n_clusters}")

# Create save directory
dec_save_dir = './Experimentally-obtained vPCF Testing/results/dec'
os.makedirs(dec_save_dir, exist_ok=True)

# Initialize DEC
dec = DEC(dims=dims, n_clusters=n_clusters, save_dir=dec_save_dir)
print("\nDEC model initialized")

In [ ]:
# Pretrain autoencoder
print("Pretraining autoencoder...")
dec.pretrain(x, epochs=50, batch_size=256)

In [ ]:
# Train clustering layer
print("Training clustering layer...")
dec.compile(optimizer='sgd')

dec_labels = dec.fit(
    x,
    y=None,
    maxiter=2000,  # Increase to 8000+ for full training
    update_interval=140,
    batch_size=256
)

print(f"\nDEC training complete!")
print(f"Cluster distribution: {np.bincount(dec_labels)}")

## Step 4: Train IDEC Model

In [ ]:
from src.IDEC import IDEC

# Create save directory
idec_save_dir = './Experimentally-obtained vPCF Testing/results/idec'
os.makedirs(idec_save_dir, exist_ok=True)

# Initialize IDEC
idec = IDEC(
    dims=dims,
    n_clusters=n_clusters,
    gamma=0.1,
    save_dir=idec_save_dir
)
print("IDEC model initialized")

In [ ]:
# Pretrain autoencoder
print("Pretraining autoencoder...")
idec.pretrain(x, epochs=50, batch_size=256)

In [ ]:
# Train clustering layer
print("Training clustering layer...")
idec.compile(optimizer='sgd')

idec_labels = idec.fit(
    x,
    y=None,
    maxiter=2000,  # Increase to 8000+ for full training
    update_interval=140,
    batch_size=256
)

print(f"\nIDEC training complete!")
print(f"Cluster distribution: {np.bincount(idec_labels)}")

## Step 5: Evaluate Metrics

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

print("DEC Metrics:")
print(f"  Silhouette Score: {silhouette_score(x, dec_labels):.4f}")
print(f"  Davies-Bouldin Index: {davies_bouldin_score(x, dec_labels):.4f}")
print(f"  Calinski-Harabasz Index: {calinski_harabasz_score(x, dec_labels):.4f}")

print("\nIDEC Metrics:")
print(f"  Silhouette Score: {silhouette_score(x, idec_labels):.4f}")
print(f"  Davies-Bouldin Index: {davies_bouldin_score(x, idec_labels):.4f}")
print(f"  Calinski-Harabasz Index: {calinski_harabasz_score(x, idec_labels):.4f}")

## Step 6: Save Results

In [ ]:
import pandas as pd

# Save cluster assignments
results_df = pd.DataFrame({
    'sample_idx': np.arange(len(dec_labels)),
    'dec_cluster': dec_labels,
    'idec_cluster': idec_labels
})

results_df.to_csv('./Experimentally-obtained vPCF Testing/results/cluster_assignments.csv', index=False)
print("Saved cluster assignments")

# Cluster agreement
agreement = (dec_labels == idec_labels).mean() * 100
print(f"\nCluster agreement: {agreement:.2f}%")

## Step 7: Visualize Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# DEC cluster distribution
dec_counts = np.bincount(dec_labels, minlength=n_clusters)
axes[0].bar(range(n_clusters), dec_counts)
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Count')
axes[0].set_title('DEC Cluster Distribution')

# IDEC cluster distribution
idec_counts = np.bincount(idec_labels, minlength=n_clusters)
axes[1].bar(range(n_clusters), idec_counts)
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Count')
axes[1].set_title('IDEC Cluster Distribution')

plt.tight_layout()
plt.savefig('./Experimentally-obtained vPCF Testing/results/cluster_comparison.png', dpi=150)
plt.show()

print("Saved cluster comparison plot")

## Step 8: Download Results (if on Colab)

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    results_dir = './Experimentally-obtained vPCF Testing/results'
    
    # Download all results
    print("Files to download:")
    for f in os.listdir(results_dir):
        print(f"  {f}")
    
    print("\nDownloading results...")
    # Zip and download
    import shutil
    shutil.make_archive('vpcf_results', 'zip', results_dir)
    files.download('vpcf_results.zip')
    print("Download complete!")
else:
    print("Results saved to: ./Experimentally-obtained vPCF Testing/results")